# Quasi-Experiments

Quasi-experiments are research designs that aim to estimate causal effects without full randomization - they occupy the space between a controlled experiment (where you randomly assign treatment) and pure observation (where you simply watch what happens). Here's a comprehensive treatment.

## What Is a Quasi-Experiment?

In a **true experiment**, the researcher controls who gets treated by randomizing assignment. This makes treated and control groups statistically identical in expectation, so any difference in outcomes is attributable to the treatment.

In a **quasi-experiment**, treatment assignment is *not* under the researcher's control and is *not* random - but it has enough *structure* that, under defensible assumptions, we can still recover a causal estimate. The word "quasi" (Latin: *as if*) captures exactly this: we treat part of the data *as if* it had been randomized, even though it wasn't.

The defining feature is the **identifying assumption** - a claim about the data-generating process that, if true, makes the non-random assignment equivalent to random assignment for the purpose of estimating a specific causal effect. The craft of quasi-experimentation is choosing the assumption that best fits the known facts about *how* the intervention was actually assigned, and then attacking that assumption aggressively with falsification tests.

## The Counterfactual Logic

Every causal question is a question about a **counterfactual**: what would have happened to the treated units if they had *not* been treated? We never observe both states for the same unit at the same time - this is the fundamental problem of causal inference (Holland 1986). Quasi-experiments are systematic strategies for constructing a credible counterfactual from available data.

The estimand - *what exactly* we are trying to measure - matters:

- **ATE (Average Treatment Effect):** the effect if everyone in the population were treated.
- **ATT (Average Treatment Effect on the Treated):** the effect for the units that actually received the intervention. Most business rollout evaluations target the ATT.
- **LATE (Local Average Treatment Effect):** the effect for the subgroup whose treatment status was moved by some external push (an instrument). Relevant in Section 5 below.
- **CATE (Conditional ATE):** the effect for a specific subgroup defined by covariates - the "for whom" question that drives targeting.

Being explicit about the estimand prevents a common stakeholder confusion: a synthetic control estimates the ATT for one treated market; an IV estimates a LATE that may be very different from the ATE.

## The Family of Methods and Their Assumptions

Each quasi-experimental design makes a different identifying assumption, suited to a different kind of assignment mechanism.

### 1. Difference-in-Differences (DiD)

**Assignment mechanism:** treatment happened to some groups (markets, stores, customer segments) at a known point in time, while other groups remained untreated throughout.

**Counterfactual construction:** the untreated group's trajectory over time serves as the counterfactual for the treated group. DiD estimates the treatment effect as the change in the treated group's outcome minus the change in the untreated group's outcome over the same period - "the difference in differences."

**Identifying assumption - parallel trends:** absent the treatment, the treated and control groups' outcomes would have followed the same trajectory (they need not start at the same level). This cannot be proven - it is a claim about an unobservable counterfactual - but it can be probed by checking that the groups moved in parallel during the pre-treatment period, and by demonstrating that the divergence occurs precisely at the intervention date.

**When it breaks:** if the treated group was trending differently from the control group before treatment (e.g., a struggling division gets the intervention, or a thriving region gets the new store format first), parallel trends fails and DiD is biased. It also breaks when something *else* happens to only the treated group at the same time as the intervention - a common failure in business settings where events cluster geographically.

**Modern complications - staggered rollouts:** in practice, business programs roll out in waves, not all at once. A large econometrics literature (Goodman-Bacon 2021; Callaway & Sant'Anna 2021; Sun & Abraham 2021) showed that the standard two-way fixed-effects regression misidentifies the effect in staggered settings with dynamic treatment effects, because it uses already-treated units as controls - introducing "forbidden comparisons" with potentially negative weights. Modern practice uses heterogeneity-robust estimators (Callaway–Sant'Anna `did` package in R, `csdid` in Python) that restrict comparisons to never-treated or not-yet-treated units.

### 2. Event Study / Dynamic DiD

An extension of DiD that replaces the single "before/after" estimate with a full sequence of estimates at each time point relative to the intervention. This serves two purposes simultaneously: it is a **pre-trend test** (did the groups diverge before treatment? coefficients should be near zero), and it reveals the **dynamic trajectory** of the effect (does it build over time? decay? spike and revert?). The event-study plot is often the single most persuasive artifact in a quasi-experimental evaluation because it makes both the identification strategy and the effect's trajectory immediately visible.

### 3. Interrupted Time Series (ITS)

**Assignment mechanism:** the intervention applied to everyone at once, with no contemporaneous control group. You have only the outcome's time series before and after.

**Counterfactual construction:** the pre-intervention trend (including seasonality and any covariates) is extrapolated forward as the counterfactual. The effect is the deviation from that projection after the intervention.

**Identifying assumption:** the pre-period model would have continued unchanged absent the intervention. This is a strong and fragile assumption - anything that co-occurs with the intervention (another policy change, a seasonal shift, a competitor exit) is indistinguishable from the effect. ITS is the weakest design in this family on its own, but is greatly strengthened by adding a **control series** (a comparable product line or region not exposed to the intervention), which restores the DiD logic with rich temporal structure.

### 4. Bayesian Structural Time Series / CausalImpact

A probabilistic generalization of ITS, developed at Google (Brodersen et al. 2015). Instead of a single extrapolated trend, a Bayesian state-space model is fitted to the treated series using both its own structure and a set of contemporaneous donor series (other markets, categories, or metrics that were not affected). The posterior distribution over the counterfactual propagates uncertainty naturally, and the cumulative effect with credible intervals is the output. This is particularly well-suited to cases with one treated unit and several plausible - but imperfect - control series, where standard DiD's parallel-trends assumption is difficult to justify for any single control.

### 5. Synthetic Control Method

**Assignment mechanism:** one (or a small number of) treated units at a known time, with a pool of untreated donors.

**Counterfactual construction:** a weighted average of donor units - with nonnegative weights summing to 1 - is constructed to match the treated unit's pre-intervention outcome path (and optionally other predictors) as closely as possible. The synthetic unit is the counterfactual; the post-period gap between the treated unit and its synthetic twin is the effect.

**Identifying assumption:** the pre-period weighting that reproduces the treated unit's history would have continued to reproduce it absent treatment. This is more transparent than parallel trends: you can see how well the synthetic unit tracks the treated unit in the pre-period, and that fit quality is the license to interpret the post-period gap.

**Inference** uses permutation: the same procedure is applied to every donor as if it were treated, and the distribution of these placebo gaps calibrates how unusual the real effect is - an exact randomization test without any distributional assumptions. The method's transparency ("Synthetic Chicago = 40% Minneapolis + 35% Detroit + 25% Cleveland") makes it unusually persuasive to non-technical audiences and resistant to specification-curve criticism.

### 6. Propensity Score Methods (Matching and Weighting)

**Assignment mechanism:** treatment is self-selected by individual units - customers who opted into a program, stores whose managers chose to participate, merchants who adopted a new feature.

**Counterfactual construction:** statistical adjustment for observed pre-treatment covariates to make treated and control units comparable. Matching pairs each treated unit with a control unit that has a similar probability of treatment (the propensity score); weighting reweights the control group to resemble the treated population.

**Identifying assumption - conditional ignorability:** given the observed covariates, treatment is as good as random. Equivalently: there are no *unmeasured* variables that simultaneously drive treatment uptake and the outcome. This is strong and untestable. Its credibility depends entirely on how rich and relevant the covariate set is - which is why the lagged outcome is the single most important covariate to include (it absorbs most pre-existing differences).

**Doubly robust estimation** (AIPW) combines an outcome regression with propensity weighting and is consistent if *either* model is correct - the recommended default for modern practice.

### 7. Regression Discontinuity Design (RDD)

**Assignment mechanism:** treatment is determined by whether a continuous running variable crosses a known threshold. This structure appears constantly in business: risk scores, credit ratings, performance ratings, order values (free shipping above $50), customer tiers.

**Counterfactual construction:** units just above and just below the cutoff are compared. Under the assumption that nothing else jumps discontinuously at the threshold, these units are essentially identical except for treatment assignment, and the comparison is locally as good as randomized.

**Identifying assumption:** potential outcomes vary smoothly through the cutoff. This means (a) no other policy or variable also changes at that exact threshold, and (b) units cannot precisely manipulate their score to sort across the cutoff. Both are testable: covariate continuity checks test (a); the McCrary density test tests (b) by checking whether the density of the running variable has a suspicious spike just above the threshold.

**What it identifies:** strictly a *local* effect - the treatment effect for units near the cutoff, not the population average. This is exactly the right quantity for the marginal decision ("should we raise or lower the threshold?") but should not be extrapolated to the full population without argument.

### 8. Instrumental Variables (IV)

**Assignment mechanism:** treatment is self-selected and confounded by unobservables (things that drive both treatment uptake and the outcome that you cannot measure), but some variable - the instrument - shifts treatment probability without directly affecting the outcome.

**Counterfactual construction:** the instrument creates exogenous variation in treatment assignment that "mimics" randomization for the complier subgroup. The treatment effect is recovered by scaling the instrument's reduced-form effect on the outcome by its effect on treatment uptake.

**Identifying assumptions:** the instrument must be (1) relevant (it must actually move treatment, testable via the first-stage F-statistic), (2) excludable (it affects the outcome *only through* its effect on treatment, not testable but must be argued from institutional knowledge), and (3) as good as random with respect to potential outcomes.

**What it identifies - the LATE:** the effect specifically on *compliers* - units whose treatment status was changed by the instrument. If the instrument is a randomly sent encouragement email, compliers are customers who adopted *because of* the email; never-takers and always-takers are excluded. The LATE may be a very different quantity from the ATE.

### 9. Double Machine Learning (DML)

A modern extension of propensity-score methods that uses machine learning to estimate the nuisance functions (the propensity score and the outcome model) flexibly, then applies a cross-fitting procedure to prevent regularization bias from leaking into the treatment effect estimate. The identifying assumption is the same as standard propensity methods (conditional ignorability), but DML allows that assumption to hold in a high-dimensional covariate space - hundreds of RFM features, behavioral signals, and interaction terms - without misspecification bias from the parametric constraints of logistic regression.

## Comparison Across Designs

| Design | Identifying assumption | Testable pre-checks | Required data structure |
|---|---|---|---|
| DiD / TWFE | Parallel trends (levels or logs) | Pre-period co-movement, event study leads | Repeated observations, pre + post |
| Staggered DiD (C&S, S&A) | Parallel trends + no treated-as-control | Pre-trend test per cohort | Staggered treatment dates |
| ITS | Pre-period model continues | Placebo dates, co-intervention audit | Time series only |
| CausalImpact / BSTS | Predictive relationship persists, no spillovers | Pre-period fit, placebo-in-space | Time series + donor series |
| Synthetic control | Donor-weighted path is valid counterfactual | Pre-period fit quality | One treated unit, donor pool |
| Propensity methods | No unmeasured confounders, overlap | Balance (SMD), overlap plots | Cross-sectional or panel + covariates |
| RDD (sharp) | Continuity of potential outcomes at cutoff | Density test, covariate continuity, placebo cutoffs | Running variable + outcome |
| IV / 2SLS | Exclusion restriction, relevance, independence | First-stage F, covariate balance on Z | Instrument + treatment + outcome |
| DML | Conditional ignorability in high dimensions | Balance, nuisance model quality | Rich covariates, cross-sectional or panel |

## Why Quasi-Experiments Matter in Business

### The practical barriers to randomization

Business leaders often assume that if an experiment matters enough, it can be randomized. In practice this is wrong far more often than practitioners acknowledge.

**Legal and competitive constraints.** Charging different prices to different customers for the same product at the same time can violate price-discrimination law in some jurisdictions and is commercially toxic if discovered. Withholding a beneficial product improvement from customers in a control group raises both ethical and PR risks.

**Unit of randomization mismatch.** A television ad campaign, a billboard, a press release, or a trade show cannot be randomized at the individual level - they operate at geographic market or media-market scale. With 12 markets, a randomized experiment has very little statistical power; the uncertainty swamps the signal.

**Organizational politics.** Holding back a high-profile initiative from half the regions to create a control group is often politically impossible: the regions that don't receive the new program lobby against the experiment, and executives who championed the initiative interpret the control condition as a sign of doubt.

**Timeline pressure.** A proper randomized experiment in a retail setting takes months to design, execute, and analyze. Leadership wants an answer about last quarter's price increase *now*, not after running a prospective study.

**Spillovers and interference.** In two-sided marketplaces, social networks, and supply chains, treating one customer or store affects others nearby - violating the stable unit treatment value assumption that underpins simple A/B tests. Quasi-experimental designs can model or exploit the spatial and temporal structure of spillovers in ways that unit-randomized experiments cannot.

### Where each design naturally appears

**Retail and consumer goods.** New store formats or loyalty programs roll out regionally in waves → staggered DiD or synthetic control. National pricing changes have no holdout → ITS with regional control series. Churn-prevention calls triggered by a risk model threshold → RDD.

**Financial services.** Credit products with hard eligibility cutoffs (FICO score, debt-to-income ratio) → sharp RDD. Product upsells randomized at the branch level but with branch-level confounders → IV using branch assignment.

**Technology and platforms.** A/B testing is the norm, but geographic experiments (country-level rollouts), policy changes (content moderation rules), and algorithm changes that apply globally → synthetic control or CausalImpact. Self-selected feature adoption → DML with rich telemetry.

**Healthcare and pharmaceuticals.** Drug adoption driven by physician practice patterns → IV using physician fixed effects or regional market shares. Insurance program changes → DiD using states without the change as controls.

**Supply chain and operations.** A new fulfillment center opens in a region → DiD using similar regions without new capacity. A vendor quality program applies to all suppliers above a spend threshold → RDD.

**Human resources.** A management training program offered to all employees in certain regions → DiD; offered to employees above a performance rating threshold → RDD; offered but not mandated (managers chose whether to send their teams) → IV using the regional rollout as an instrument.

## The Hierarchy of Evidence and Practical Combinations

Not all quasi-experimental evidence is equally persuasive. Roughly:

**Stronger designs** have assignment mechanisms that are transparent, externally determined, and plausibly unrelated to the outcome except through the treatment channel. An instrument that was genuinely randomly assigned (an encouragement lottery, a natural disaster, a quasi-random legal change) is strong. An RDD where the running variable is measured by a third party and the cutoff was set by regulation is strong. A DiD where the treated and control groups were trending identically for two years before a sudden exogenous change is strong.

**Weaker designs** rely on assumptions that are hard to defend and easy to violate. ITS with a short pre-period in a volatile series is weak. Propensity score matching without a lagged outcome in the covariate set is weak. An IV whose exclusion restriction requires strong institutional arguments is only as strong as those arguments.

The best business evaluation practice **triangulates** - applies two or more designs with different identifying assumptions to the same question, and treats agreement as much stronger evidence than any single approach. If DiD, synthetic control, and CausalImpact (all with different assumptions about what the counterfactual is) converge on a +$8k/week estimate, confidence in that number is substantially higher than if only one design is available.

A second principle: **design beats estimation**. The highest-leverage moment in a business quasi-experiment is often before a program launches, when a data scientist can persuade leadership to stagger the rollout geographically, retain a small holdout region, or randomize the *encouragement* even if treatment itself cannot be randomized. A few hours of effort at the design stage creates a dramatically cleaner identification strategy and reduces the burden on untestable assumptions in the analysis.

## The Sensitivity Analysis Imperative

Because identifying assumptions are unverifiable, a credible quasi-experimental analysis always includes a systematic attempt to *break* its own conclusions:

- **Placebo-in-time:** rerun the analysis with a fake treatment date in the pre-period; you should find no effect.
- **Placebo-in-space:** rerun the analysis treating each control unit as if it had been treated; the real estimate should be an outlier relative to the placebo distribution.
- **Placebo outcome:** rerun on a metric the intervention logically could not have affected; finding an effect there signals confounding or model misspecification.
- **Specification curve:** vary the reasonable modeling choices (bandwidth, matching algorithm, control set, functional form) and show that the estimate is stable across the curve.
- **Sensitivity to unmeasured confounding:** use tools like `sensemakr` (R) or E-values to quantify *how strong* an unmeasured confounder would have to be to overturn the result, and compare that to what is known about the likely confounders in the domain.

The deliverable is not a single number but a *range of evidence*: "across all reasonable specifications and falsification checks, the estimate is consistently between +$6k and +$10k/week, and no placebo test finds a spurious effect."

## Summary

Quasi-experiments are how causal inference gets done when randomization is impractical, retroactive, or ethically unavailable - which, in real business settings, is most of the time. They are not inferior substitutes for randomized trials; they are distinct tools suited to different assignment mechanisms, and when applied with methodological discipline they can produce estimates that are credibly causal. The discipline consists of three things: choosing the design whose identifying assumption fits the known facts of how the intervention was assigned, stress-testing that assumption with every falsification test available, and being explicit with stakeholders about what the design can and cannot claim.